In [4]:
from embedder import Embedder

In [5]:
embed = Embedder()

In [24]:
# Q1. Embedding a query
q = 'How does approximate nearest neighbor search work?'
v = embed.encode(q)
v[0]

np.float64(-0.02058203437252893)

In [25]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [26]:
# Q2. Cosine similarity
document = next((x for x in documents if x['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md'), None)
dv = embed.encode(document['content'])
dv.dot(v)

np.float64(0.36107026789538205)

In [27]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [28]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    batch_vectors = embed.encode_batch([item['content'] for item in batch])
    X.extend(batch_vectors)

X = np.array(X)

scores = X.dot(v)

  0%|          | 0/6 [00:00<?, ?it/s]

In [35]:
# Q3. Chunking and search by hand
scores = X.dot(v)
idx = np.argmax(scores)

chunks[idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [40]:
# Q4. Vector search with minsearch
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

In [42]:
# Q5. Text search vs vector search
from minsearch import Index
index = Index(
    text_fields=["content"]
)
index.fit(chunks)

query = "How do I store vectors in PostgreSQL?"

results = index.search(query)
results[0]['filename']

'02-vector-search/lessons/02-embeddings.md'

In [45]:
# Q6. Hybrid search
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


query = "How do I give the model access to tools?"

query_vector = embed.encode(query)
vector_results = vindex.search(query_vector, num_results=5)

text_results = index.search(query)

results = rrf([vector_results, text_results])
results[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'